In [53]:
import torch
import torch.nn as nn
import h5py
import numpy as np
import scipy.spatial
import os


model_path = "/Users/moustholmes/Projects/METAL-AI/best_models/effect_all_2968.ckpt"
weights_path = "/Users/moustholmes/Projects/METAL-AI/best_models/clean_model_weights.pt"

In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Callable
class simple_transformer_encoder_model(nn.Module):
    def __init__(
        self,
        csf_encoder,
        input_size: int = 4, # number of allowed excitations + 2
        d_model: int = 64,
        nhead: int = 8,
        dim_forward: int = 64,
        num_layers: int = 6,
        output_size: int =1,
        dropout: float = 0.5,
        output_activation: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    ):
        super(simple_transformer_encoder_model, self).__init__()
        self.csf_encoder = csf_encoder
        encoder_layers = nn.TransformerEncoderLayer(
            self.csf_encoder.output_size, nhead, dim_forward, dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.decoder = nn.Linear(self.csf_encoder.output_size, output_size)
        self.output_activation = output_activation

    def forward(self, input_dict):
        encoded_csfs = self.csf_encoder(input_dict)
        output = self.transformer_encoder(encoded_csfs)#,src_key_padding_mask = input_dict['pad_mask']  src_key_padding_mask = ~input_dict['mask']
        # output = output[:, 0, :]
        output = self.decoder(output)

        if self.output_activation:
            output = self.output_activation(output)

        return output.squeeze(-1)

class simple_CSF_encoder(nn.Module):
    def __init__(self, output_size=32):
        super(simple_CSF_encoder, self).__init__()
        # Assuming the input size is 6 because we append n_electrons (1) and n_protons (1) to each 4-dimensional CSF.
        self.output_size = output_size
        self.network = nn.Sequential(
        nn.Linear(4, 64), # number of allowed excitations + 2
        nn.ReLU(),
        nn.Linear(64, output_size)
        )

    def forward(self, input_dict):
        excitations = input_dict["excitations"]
        n_electrons = input_dict["n_electrons"]
        n_protons = input_dict["n_protons"]

        # Append n_electrons and n_protons to each excitations
        n_electrons = n_electrons.float().unsqueeze(-1).unsqueeze(-1).expand(-1, excitations.size(1), 1)
        n_protons = n_protons.float().unsqueeze(-1).unsqueeze(-1).expand(-1, excitations.size(1), 1)
        extended_excitations = torch.cat([excitations, n_electrons, n_protons], dim=-1)
        return self.network(extended_excitations)

checkpoint_path = "/Users/moustholmes/Projects/METAL-AI/best_models/effect_all_2968.ckpt"
model = simple_transformer_encoder_model(
    simple_CSF_encoder( output_size=64),
    d_model=32,
    nhead=8,
    dim_forward=64,
    num_layers=8,
    dropout=0.0,
    output_activation=nn.ReLU(),
    output_size=1
    )

state_dict = torch.load(weights_path, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)

# Set the model to evaluation mode
model.eval()


batch_size = 3
n_asfs = 10
# Example input dictionary
example_input = {
    "excitations": torch.tensor(([[[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 1.],
         [1., 5.],
         [0., 5.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [4., 3.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 3.],
         [2., 2.],
         [2., 1.],
         [1., 5.],
         [1., 3.],
         [1., 2.],
         [0., 5.],
         [0., 4.],
         [0., 0.]],

        [[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [4., 3.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 1.],
         [1., 5.],
         [1., 2.],
         [0., 5.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]])),
    "n_electrons": torch.tensor([13, 13, 13]),
    "n_protons": torch.tensor([16, 16, 16]),
}

# Perform a forward pass
output = model(example_input)
print(output)

RuntimeError: Error(s) in loading state_dict for simple_transformer_encoder_model:
	Missing key(s) in state_dict: "csf_encoder.network.0.weight", "csf_encoder.network.0.bias", "csf_encoder.network.2.weight", "csf_encoder.network.2.bias", "transformer_encoder.layers.0.self_attn.in_proj_weight", "transformer_encoder.layers.0.self_attn.in_proj_bias", "transformer_encoder.layers.0.self_attn.out_proj.weight", "transformer_encoder.layers.0.self_attn.out_proj.bias", "transformer_encoder.layers.0.linear1.weight", "transformer_encoder.layers.0.linear1.bias", "transformer_encoder.layers.0.linear2.weight", "transformer_encoder.layers.0.linear2.bias", "transformer_encoder.layers.0.norm1.weight", "transformer_encoder.layers.0.norm1.bias", "transformer_encoder.layers.0.norm2.weight", "transformer_encoder.layers.0.norm2.bias", "transformer_encoder.layers.1.self_attn.in_proj_weight", "transformer_encoder.layers.1.self_attn.in_proj_bias", "transformer_encoder.layers.1.self_attn.out_proj.weight", "trans

In [55]:
checkpoint_path = "/Users/moustholmes/Projects/METAL-AI/best_models/epoch_2968.ckpt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

ModuleNotFoundError: No module named 'src.models.components.Transformer_encoder_model'; 'src.models.components' is not a package

In [41]:
checkpoint_path = "/Users/moustholmes/Projects/METAL-AI/best_models/epoch_2968.ckpt"
# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))  # Use 'cuda' if using GPU

# Load state dictionary into your model
model.load_state_dict(checkpoint['model_state_dict'])  # If your checkpoint saves only the model state dict
# OR
model.load_state_dict(checkpoint)  # If your checkpoint directly saves the state dict

# Set the model to evaluation mode
model.eval()

# Now you can perform inference
with torch.no_grad():
    # Prepare your input data
    input_data = {
        "excitations": torch.tensor(...),  # Your excitation data
        "n_electrons": torch.tensor(...),  # Number of electrons
        "n_protons": torch.tensor(...)     # Number of protons
    }
    
    # Forward pass
    output = model(input_data)
    
    # Process the output as needed
    # ...

ModuleNotFoundError: No module named 'src'

In [42]:
# Load just the state dict values, ignoring the class structure
import torch
import io
import pickle

# Custom function to skip module import errors
def load_tensors_only(filename, map_location='cpu'):
    loaded_state_dict = {}
    
    # Load the file in binary mode
    with open(filename, 'rb') as f:
        try:
            # Try standard loading first
            checkpoint = torch.load(filename, map_location=map_location)
            if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
                return checkpoint['state_dict']
            return checkpoint
        except ModuleNotFoundError:
            # If that fails, use a different approach
            # Reset file pointer
            f.seek(0)
            
            # Try to load using pickle directly with custom error handling
            try:
                magic_number = pickle.load(f)
                protocol_version = pickle.load(f)
                sys_info = pickle.load(f)
                
                # Continue loading with custom error handling
                while True:
                    try:
                        key = pickle.load(f)
                        tensor = pickle.load(f)
                        if isinstance(tensor, torch.Tensor):
                            loaded_state_dict[key] = tensor
                    except EOFError:
                        break
                    except:
                        # Skip this entry if there's an error
                        continue
            except:
                print("Failed to extract tensors manually.")
                
    return loaded_state_dict

# Try loading just the tensors
checkpoint = load_tensors_only(checkpoint_path)

# Check what we got
print(type(checkpoint))
print(list(checkpoint.keys()) if isinstance(checkpoint, dict) else "Not a dictionary")

# Apply the parameters that match your model structure
model_dict = model.state_dict()
if isinstance(checkpoint, dict):
    # Filter to only matching keys
    pretrained_dict = {k: v for k, v in checkpoint.items() if k in model_dict}
    print(f"Loaded {len(pretrained_dict)}/{len(model_dict)} parameters")
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

Failed to extract tensors manually.
<class 'dict'>
[]
Loaded 0/102 parameters


In [45]:
import torch
import pickle
import sys

def inspect_checkpoint_detailed(filepath):
    try:
        # First try to look at the file structure without loading classes
        with open(filepath, 'rb') as f:
            try:
                # Skip the pytorch header
                magic_number = pickle.load(f)
                protocol_version = pickle.load(f)
                sys_info = pickle.load(f)
                
                # Try to see what's next
                print("File header information loaded successfully")
                
                # Try to peek at more content
                try:
                    next_item = pickle.load(f)
                    print("Next item type:", type(next_item))
                    if isinstance(next_item, str):
                        print("String value:", next_item)
                except Exception as e:
                    print(f"Error peeking at content: {e}")
            except Exception as e:
                print(f"Error reading header: {e}")
    
        # Try to load with full PyTorch loader
        print("\nAttempting full load...")
        try:
            checkpoint = torch.load(filepath, map_location='cpu')
            print("Full load successful")
            return checkpoint
        except ModuleNotFoundError as e:
            print(f"Module not found: {e}")
            
            # Get all missing modules
            missing_module = str(e).split("'")[1] if "'" in str(e) else None
            if missing_module:
                print(f"Creating dummy module for: {missing_module}")
                
                # Create the module structure
                parts = missing_module.split('.')
                current = ''
                
                for i, part in enumerate(parts):
                    current = current + '.' + part if current else part
                    if current not in sys.modules:
                        sys.modules[current] = type(f'dummy_{part}', (), {})
                
                # Try loading again
                print("Retrying with dummy module...")
                try:
                    checkpoint = torch.load(filepath, map_location='cpu')
                    print("Load successful with dummy module")
                    return checkpoint
                except Exception as retry_e:
                    print(f"Retry failed: {retry_e}")
            
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# Run the detailed inspection
result = inspect_checkpoint_detailed(checkpoint_path)

Error reading header: A load persistent id instruction was encountered,
but no persistent_load function was specified.

Attempting full load...
Module not found: No module named 'src'
Creating dummy module for: src
Retrying with dummy module...
Retry failed: No module named 'src.models'; 'src' is not a package


In [46]:
import torch
import sys
import types

# First, create a fake 'src' module and its submodules
# This is to trick the unpickler into loading the checkpoint
src_module = types.ModuleType('src')
sys.modules['src'] = src_module

# Create any necessary submodules
models_module = types.ModuleType('src.models')
sys.modules['src.models'] = models_module

# Create dummy classes to match what's in the checkpoint
# You'll need to adapt these based on what classes your checkpoint expects
class DummyCSFEncoder:
    pass

class DummyTransformerModel:
    pass

# Add these classes to the models module
models_module.simple_CSF_encoder = simple_CSF_encoder
models_module.simple_transformer_encoder_model = simple_transformer_encoder_model

# Now try to load the checkpoint
try:
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    print("Checkpoint loaded successfully!")
    print("Type:", type(checkpoint))
    
    if isinstance(checkpoint, dict):
        print("Keys:", list(checkpoint.keys()))
    
    # Try to extract the state dict
    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        elif 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        elif 'model' in checkpoint:
            state_dict = checkpoint['model']
        else:
            # Assume the checkpoint itself is the state dict
            state_dict = checkpoint
            
        print(f"Found {len(state_dict)} parameters")
        
        # Save just the state dict to a new file
        output_path = "extracted_weights.pt"
        torch.save(state_dict, output_path)
        print(f"Saved extracted weights to {output_path}")
    
except Exception as e:
    print(f"Error: {e}")

Error: No module named 'src.models.components'; 'src.models' is not a package


In [48]:
import torch
import os

# Load the checkpoint in the original environment where all imports work
checkpoint = torch.load(checkpoint_path)

# Extract the state dictionary
if isinstance(checkpoint, dict):
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict'] 
    else:
        state_dict = checkpoint
else:
    state_dict = checkpoint

# Save just the state dictionary without any class references
torch.save(state_dict, "clean_weights.pt")
print(f"Saved clean weights to {os.path.abspath('clean_weights.pt')}")

ModuleNotFoundError: No module named 'src.models.components.Transformer_encoder_model'; 'src.models.components' is not a package

In [50]:
torch.load(checkpoint_path, map_location='cpu')

ModuleNotFoundError: No module named 'src.models.components.Transformer_encoder_model'; 'src.models.components' is not a package

In [37]:
effect = torch.tensor([[2.4899e+00, 1.1615e+01, 1.1168e+01, 2.1855e+00, 9.5257e+00, 1.1765e+01,
         1.9002e+00, 1.3462e+01, 1.8127e+00, 1.0621e+01, 9.8880e+00, 9.3574e+00,
         8.1096e+00, 9.7354e-05, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [2.3518e+00, 9.8752e+00, 1.0028e+01, 2.0473e+00, 9.4732e+00, 9.6758e+00,
         9.0311e+00, 1.8497e+00, 1.1089e+01, 1.8071e+00, 9.0798e+00, 1.7521e+00,
         8.9316e+00, 9.1691e+00, 8.8963e+00, 7.8625e+00, 9.9450e-01, 8.2250e+00,
         7.8042e+00, 2.4267e-03],
        [2.5658e+00, 1.1713e+01, 1.1264e+01, 2.2098e+00, 1.2391e+01, 9.6175e+00,
         1.1871e+01, 1.9015e+00, 1.3589e+01, 1.8045e+00, 1.0715e+01, 9.9837e+00,
         9.4531e+00, 9.7011e-01, 8.2055e+00, 2.6050e-03, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00]])

In [34]:
excitation=torch.tensor(([[[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 1.],
         [1., 5.],
         [0., 5.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [4., 3.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 3.],
         [2., 2.],
         [2., 1.],
         [1., 5.],
         [1., 3.],
         [1., 2.],
         [0., 5.],
         [0., 4.],
         [0., 0.]],

        [[5., 5.],
         [5., 4.],
         [4., 5.],
         [4., 4.],
         [4., 3.],
         [3., 5.],
         [3., 4.],
         [3., 3.],
         [3., 2.],
         [2., 5.],
         [2., 4.],
         [2., 1.],
         [1., 5.],
         [1., 2.],
         [0., 5.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]]))
excitation.shape

torch.Size([3, 20, 2])

In [27]:
batch_size = 10
torch.randint(0, 10, (batch_size, 2))

tensor([[6, 9],
        [8, 4],
        [2, 3],
        [5, 1],
        [2, 1],
        [1, 6],
        [6, 8],
        [1, 4],
        [0, 2],
        [5, 9]])

In [24]:
torch.randn(1, 10, 2)

tensor([[[ 0.1765,  0.5580],
         [ 2.9335, -1.0916],
         [ 0.5986, -0.3359],
         [ 0.0213,  1.0019],
         [-1.6912,  0.8587],
         [ 0.0380, -0.8175],
         [-1.4460,  0.7189],
         [-1.6692,  0.8332],
         [ 0.9768, -0.5746],
         [ 0.2988,  0.4431]]])

In [10]:
model = simple_transformer_encoder_model(
    simple_CSF_encoder( output_size=64),
    d_model=32,
    nhead=8,
    dim_forward=64,
    num_layers=8,
    dropout=0.0,
    output_activation=nn.ReLU(),
    output_size=1
    )

In [16]:
checkpoint_path = "/Users/moustholmes/Projects/METAL-AI/best_models/effect_all_2968.ckpt"
model = simple_transformer_encoder_model(
    simple_CSF_encoder( output_size=64),
    d_model=32,
    nhead=8,
    dim_forward=64,
    num_layers=8,
    dropout=0.0,
    output_activation=nn.ReLU(),
    output_size=1
    )

# Load the checkpoint
checkpoint = torch.load(checkpoint_path)

# Load the state dictionary into the model
model.load_state_dict(checkpoint['state_dict'])
# # load model from checkpoint
model = torch.load(model_path)

In [20]:
model

{'state_dict': OrderedDict([('csf_encoder.network.0.weight',
               tensor([[-2.4237e-01, -3.3229e-02,  2.1965e-01,  2.3920e-01],
                       [-1.5243e-02,  4.6459e-01, -1.8444e-01, -3.8284e-01],
                       [ 3.5456e-01, -3.1058e-01, -1.6190e-01,  3.9781e-01],
                       [-4.3428e-01, -2.8893e-02, -4.2506e-01, -1.5797e-02],
                       [ 4.7364e-01, -3.0093e-01, -4.2747e-01,  4.9261e-01],
                       [ 3.7045e-01,  3.0376e-02,  2.3343e-01, -4.8854e-01],
                       [ 1.4877e-01, -9.5798e-02, -2.9369e-01,  4.6373e-01],
                       [ 4.5866e-01,  4.7050e-01, -2.0690e-01, -3.4354e-01],
                       [ 4.8066e-02,  3.0434e-01,  2.1220e-01,  6.2326e-02],
                       [ 3.4456e-02, -2.4342e-01,  3.7150e-01, -3.4985e-01],
                       [ 7.4343e-02,  2.5811e-01,  3.7557e-02,  2.5969e-01],
                       [ 3.5250e-01,  3.2154e-01, -4.9732e-01,  7.9599e-02],
               

In [14]:
# Assuming `model` is your trained model
checkpoint_path = "/Users/moustholmes/Projects/METAL-AI/best_models/effect_all_2968.ckpt"
torch.save({'state_dict': model.state_dict()}, checkpoint_path)

In [17]:
example_input = {
    "excitations": torch.randn(1, 10, 4),
    "n_electrons": torch.randint(0, 10, (1,)),
    "n_protons": torch.randint(0, 10, (1,))
}
model(example_input)

TypeError: 'dict' object is not callable